# 고객 행동 인사이트 리포트 — 확장판
모두몰 데이터(`events`, `customers`, `orders`)로 직접 정한 비즈니스 질문 5개에 답합니다. BigQuery 없이 **DuckDB(in-memory)** 로 실행되도록 만들었습니다 — 셀을 그대로 실행하면 결과가 나옵니다.

**공통 가정**: `orders.status`가 `Cancelled`, `Returned`인 건은 "유효 구매"에서 제외한다 (`Placed`, `Paid`, `Shipped`만 유효 구매로 취급). 이 기준은 각 CTE 최상단에서 명시적으로 필터링한다.

**종합 실습과의 차별점**: 종합 실습(전체 퍼널 총계 / 첫 활동월 코호트+1개월 잔존율 / 고정 임계값 RFM 등급 인원 / 월별 취소반품율 / 경과일·구매횟수 기준 이탈위험 명단)과 겹치지 않도록, 아래 5개는 각각 **등급별 퍼널**, **첫 구매월 기준 코호트 곡선**, **RFM 분위수 vs 실제 grade 불일치 진단**, **국가·등급별 취소반품율**, **평균 재구매 소요일(신규 각도)+이탈위험**으로 구성했다.

## 0. 데이터 준비
실제 프로젝트에서는 BigQuery/사내 DW에서 읽어오지만, 여기서는 과제에 주어진 데이터를 DuckDB 인메모리 테이블로 그대로 구성한다.

In [1]:
import duckdb
import pandas as pd
pd.set_option('display.max_rows', 50)

con = duckdb.connect()
con.execute('''
CREATE OR REPLACE TABLE events (
  event_id INTEGER,
  customer_id VARCHAR,
  event_type VARCHAR,
  page VARCHAR,
  event_at TIMESTAMP
);

INSERT INTO events VALUES
  (1, 'C001', 'visit', 'home', TIMESTAMP '2023-09-02 10:02:11'),
  (2, 'C001', 'view', 'product_detail', TIMESTAMP '2023-09-02 10:04:35'),
  (3, 'C001', 'view', 'product_detail', TIMESTAMP '2023-09-02 10:07:20'),
  (4, 'C001', 'add_to_cart', 'cart', TIMESTAMP '2023-09-02 10:11:02'),
  (5, 'C001', 'purchase', 'checkout', TIMESTAMP '2023-09-02 10:14:47'),
  (6, 'C001', 'visit', 'home', TIMESTAMP '2023-10-18 09:31:04'),
  (7, 'C001', 'view', 'product_detail', TIMESTAMP '2023-10-18 09:33:50'),
  (8, 'C001', 'visit', 'home', TIMESTAMP '2023-11-14 20:12:33'),
  (9, 'C002', 'visit', 'home', TIMESTAMP '2023-09-05 14:20:05'),
  (10, 'C002', 'view', 'product_detail', TIMESTAMP '2023-09-05 14:22:41'),
  (11, 'C002', 'add_to_cart', 'cart', TIMESTAMP '2023-09-05 14:27:19'),
  (12, 'C002', 'purchase', 'checkout', TIMESTAMP '2023-09-05 14:31:58'),
  (13, 'C002', 'visit', 'home', TIMESTAMP '2023-10-09 11:05:22'),
  (14, 'C002', 'view', 'product_detail', TIMESTAMP '2023-10-09 11:08:44'),
  (15, 'C003', 'visit', 'home', TIMESTAMP '2023-09-15 09:14:30'),
  (16, 'C003', 'view', 'product_detail', TIMESTAMP '2023-09-15 09:17:12'),
  (17, 'C003', 'add_to_cart', 'cart', TIMESTAMP '2023-09-15 09:21:40'),
  (18, 'C003', 'purchase', 'checkout', TIMESTAMP '2023-09-15 09:26:03'),
  (19, 'C003', 'visit', 'home', TIMESTAMP '2023-11-07 21:40:15'),
  (20, 'C003', 'view', 'product_detail', TIMESTAMP '2023-11-07 21:43:02'),
  (21, 'C004', 'visit', 'home', TIMESTAMP '2023-09-18 16:03:11'),
  (22, 'C004', 'view', 'product_detail', TIMESTAMP '2023-09-18 16:06:29'),
  (23, 'C004', 'visit', 'home', TIMESTAMP '2023-11-05 10:55:07'),
  (24, 'C004', 'view', 'product_detail', TIMESTAMP '2023-11-05 10:58:33'),
  (25, 'C005', 'visit', 'home', TIMESTAMP '2023-09-22 08:40:19'),
  (26, 'C006', 'visit', 'home', TIMESTAMP '2023-10-04 13:11:02'),
  (27, 'C006', 'view', 'product_detail', TIMESTAMP '2023-10-04 13:13:47'),
  (28, 'C006', 'add_to_cart', 'cart', TIMESTAMP '2023-10-04 13:18:20'),
  (29, 'C006', 'purchase', 'checkout', TIMESTAMP '2023-10-04 13:22:55'),
  (30, 'C006', 'visit', 'home', TIMESTAMP '2023-11-16 19:02:41'),
  (31, 'C006', 'visit', 'home', TIMESTAMP '2023-12-05 12:30:18'),
  (32, 'C006', 'view', 'product_detail', TIMESTAMP '2023-12-05 12:33:04'),
  (33, 'C007', 'visit', 'home', TIMESTAMP '2023-10-12 10:45:33'),
  (34, 'C007', 'view', 'product_detail', TIMESTAMP '2023-10-12 10:48:09'),
  (35, 'C007', 'view', 'product_detail', TIMESTAMP '2023-10-12 10:52:41'),
  (36, 'C007', 'add_to_cart', 'cart', TIMESTAMP '2023-10-12 10:57:12'),
  (37, 'C007', 'purchase', 'checkout', TIMESTAMP '2023-10-12 11:02:38'),
  (38, 'C007', 'visit', 'home', TIMESTAMP '2023-11-21 15:20:07'),
  (39, 'C007', 'view', 'product_detail', TIMESTAMP '2023-11-21 15:23:55'),
  (40, 'C008', 'visit', 'home', TIMESTAMP '2023-10-16 17:30:44'),
  (41, 'C008', 'view', 'product_detail', TIMESTAMP '2023-10-16 17:33:21'),
  (42, 'C008', 'visit', 'home', TIMESTAMP '2023-12-13 11:15:02'),
  (43, 'C008', 'view', 'product_detail', TIMESTAMP '2023-12-13 11:18:39'),
  (44, 'C009', 'visit', 'home', TIMESTAMP '2023-10-25 20:05:13'),
  (45, 'C009', 'view', 'product_detail', TIMESTAMP '2023-10-25 20:08:47'),
  (46, 'C009', 'add_to_cart', 'cart', TIMESTAMP '2023-10-25 20:14:22'),
  (47, 'C010', 'visit', 'home', TIMESTAMP '2023-11-24 09:50:11'),
  (48, 'C010', 'view', 'product_detail', TIMESTAMP '2023-11-24 09:52:48'),
  (49, 'C010', 'add_to_cart', 'cart', TIMESTAMP '2023-11-24 09:58:30'),
  (50, 'C010', 'visit', 'home', TIMESTAMP '2023-12-18 18:22:05'),
  (51, 'C011', 'visit', 'home', TIMESTAMP '2023-11-10 14:08:26'),
  (52, 'C011', 'view', 'product_detail', TIMESTAMP '2023-11-10 14:11:03'),
  (53, 'C012', 'visit', 'home', TIMESTAMP '2023-11-19 11:40:15'),
  (54, 'C012', 'view', 'product_detail', TIMESTAMP '2023-11-19 11:42:58'),
  (55, 'C012', 'add_to_cart', 'cart', TIMESTAMP '2023-11-19 11:47:33'),
  (56, 'C012', 'purchase', 'checkout', TIMESTAMP '2023-11-19 11:52:10'),
  (57, 'C012', 'visit', 'home', TIMESTAMP '2023-12-27 16:14:40'),
  (58, 'C012', 'view', 'product_detail', TIMESTAMP '2023-12-27 16:17:22'),
  (59, 'C012', 'visit', 'home', TIMESTAMP '2024-01-08 10:20:55'),
  (60, 'C013', 'visit', 'home', TIMESTAMP '2023-11-26 13:05:30'),
  (61, 'C013', 'view', 'product_detail', TIMESTAMP '2023-11-26 13:08:12'),
  (62, 'C013', 'visit', 'home', TIMESTAMP '2024-01-19 09:44:18'),
  (63, 'C013', 'view', 'product_detail', TIMESTAMP '2024-01-19 09:47:01'),
  (64, 'C014', 'visit', 'home', TIMESTAMP '2023-12-11 19:30:22'),
  (65, 'C014', 'visit', 'home', TIMESTAMP '2024-02-06 08:15:44'),
  (66, 'C015', 'visit', 'home', TIMESTAMP '2023-12-22 10:10:05'),
  (67, 'C015', 'view', 'product_detail', TIMESTAMP '2023-12-22 10:13:40'),
  (68, 'C015', 'visit', 'home', TIMESTAMP '2024-01-11 17:55:19');

CREATE OR REPLACE TABLE customers (
  customer_id VARCHAR,
  name VARCHAR,
  country VARCHAR,
  signup_date DATE,
  grade VARCHAR
);
INSERT INTO customers VALUES
  ('C001', '김민준', 'Korea', '2023-01-05', 'Gold'),
  ('C002', '이서연', 'Korea', '2023-02-11', 'Silver'),
  ('C003', '박도윤', 'Japan', '2023-02-20', 'Bronze'),
  ('C004', '최지우', 'USA', '2023-03-03', 'Gold'),
  ('C005', '정하준', 'Korea', '2023-03-15', 'Silver'),
  ('C006', '강서윤', 'Korea', '2023-04-01', 'Bronze'),
  ('C007', '조은우', 'Japan', '2023-04-18', 'Silver'),
  ('C008', '윤지호', 'USA', '2023-05-09', 'Gold'),
  ('C009', '임하은', 'Korea', '2023-05-22', 'Bronze'),
  ('C010', '한예준', 'Korea', '2023-06-02', 'Silver'),
  ('C011', '오시우', NULL, '2023-06-19', 'Bronze'),
  ('C012', '신아린', 'Japan', '2023-07-07', 'Silver'),
  ('C013', '권준서', 'Korea', '2023-07-25', 'Gold'),
  ('C014', '황지안', 'USA', '2023-08-10', NULL),
  ('C015', '안수아', 'Korea', '2023-08-28', 'Bronze');

CREATE OR REPLACE TABLE orders (
  order_id VARCHAR,
  customer_id VARCHAR,
  order_date DATE,
  status VARCHAR,
  amount DECIMAL(12,2)
);
INSERT INTO orders VALUES
  ('O0001', 'C001', '2023-09-02', 'Paid', 125000),
  ('O0002', 'C002', '2023-09-05', 'Shipped', 89000),
  ('O0003', 'C001', '2023-09-11', 'Returned', 45000),
  ('O0004', 'C003', '2023-09-15', 'Paid', 230000),
  ('O0005', 'C004', '2023-09-20', 'Cancelled', NULL),
  ('O0006', 'C005', '2023-09-25', 'Shipped', 67000),
  ('O0007', 'C002', '2023-10-01', 'Paid', 158000),
  ('O0008', 'C006', '2023-10-04', 'Placed', 32000),
  ('O0009', 'C007', '2023-10-12', 'Shipped', 410000),
  ('O0010', 'C008', '2023-10-19', 'Paid', 99000),
  ('O0011', 'C001', '2023-10-23', 'Paid', 76000),
  ('O0012', 'C009', '2023-10-28', 'Cancelled', NULL),
  ('O0013', 'C010', '2023-11-02', 'Shipped', 142000),
  ('O0014', 'C004', '2023-11-08', 'Paid', 88000),
  ('O0015', 'C011', '2023-11-13', 'Placed', 53000),
  ('O0016', 'C012', '2023-11-19', 'Shipped', 175000),
  ('O0017', 'C002', '2023-11-24', 'Returned', 61000),
  ('O0018', 'C013', '2023-11-29', 'Paid', 320000),
  ('O0019', 'C005', '2023-12-03', 'Paid', 47000),
  ('O0020', 'C008', '2023-12-09', 'Shipped', 215000),
  ('O0021', 'C014', '2023-12-14', 'Placed', 38000),
  ('O0022', 'C001', '2023-12-20', 'Paid', 134000),
  ('O0023', 'C015', '2023-12-25', 'Shipped', 92000),
  ('O0024', 'C007', '2024-01-03', 'Paid', 268000),
  ('O0025', 'C010', '2024-01-09', 'Cancelled', NULL),
  ('O0026', 'C003', '2024-01-15', 'Paid', 119000),
  ('O0027', 'C013', '2024-01-22', 'Shipped', 405000),
  ('O0028', 'C006', '2024-01-28', 'Paid', 58000),
  ('O0029', 'C004', '2024-02-04', 'Returned', 73000),
  ('O0030', 'C012', '2024-02-11', 'Paid', 187000);
''')

print('events:', con.execute('select count(*) from events').fetchone()[0])
print('customers:', con.execute('select count(*) from customers').fetchone()[0])
print('orders:', con.execute('select count(*) from orders').fetchone()[0])


events: 68
customers: 15
orders: 30


---
## Q1. 등급(Gold/Silver/Bronze)별로 퍼널 전환율이 다른가? (퍼널)

종합 실습 Q1은 전체 고객을 합친 총계 퍼널이었다. 여기서는 `customers.grade`로 쪼개서, 어느 등급에서 어떤 단계가 새는지를 본다.

In [2]:
q1 = """
WITH funnel_steps AS (
  SELECT
    e.customer_id,
    c.grade,
    MAX(CASE WHEN e.event_type = 'visit' THEN 1 ELSE 0 END) AS did_visit,
    MAX(CASE WHEN e.event_type = 'view' THEN 1 ELSE 0 END) AS did_view,
    MAX(CASE WHEN e.event_type = 'add_to_cart' THEN 1 ELSE 0 END) AS did_cart,
    MAX(CASE WHEN e.event_type = 'purchase' THEN 1 ELSE 0 END) AS did_purchase
  FROM events e
  LEFT JOIN customers c ON e.customer_id = c.customer_id
  GROUP BY e.customer_id, c.grade
),
funnel_by_grade AS (
  SELECT
    COALESCE(grade, '미상') AS grade,
    COUNT(DISTINCT CASE WHEN did_visit = 1 THEN customer_id END) AS step1_visit,
    COUNT(DISTINCT CASE WHEN did_visit = 1 AND did_view = 1 THEN customer_id END) AS step2_view,
    COUNT(DISTINCT CASE WHEN did_visit = 1 AND did_view = 1 AND did_cart = 1 THEN customer_id END) AS step3_cart,
    COUNT(DISTINCT CASE WHEN did_visit = 1 AND did_view = 1 AND did_cart = 1 AND did_purchase = 1 THEN customer_id END) AS step4_purchase
  FROM funnel_steps
  GROUP BY grade
)
SELECT
  grade,
  step1_visit, step2_view, step3_cart, step4_purchase,
  ROUND(step2_view * 1.0 / NULLIF(step1_visit, 0) * 100, 1)     AS visit_to_view_pct,
  ROUND(step3_cart * 1.0 / NULLIF(step2_view, 0) * 100, 1)      AS view_to_cart_pct,
  ROUND(step4_purchase * 1.0 / NULLIF(step3_cart, 0) * 100, 1)  AS cart_to_purchase_pct,
  ROUND(step4_purchase * 1.0 / NULLIF(step1_visit, 0) * 100, 1) AS overall_conversion_pct
FROM funnel_by_grade
ORDER BY step1_visit DESC
"""
df_q1 = con.execute(q1).df()
df_q1


,grade,step1_visit,step2_view,step3_cart,step4_purchase,visit_to_view_pct,view_to_cart_pct,cart_to_purchase_pct,overall_conversion_pct
0,Bronze,5,5,3,2,100.0,60.0,66.7,40.0
1,Silver,5,4,4,3,80.0,100.0,75.0,60.0
2,Gold,4,4,1,1,100.0,25.0,100.0,25.0
3,미상,1,0,0,0,0.0,NaN,NaN,0.0


**결과**: Gold(4명)의 조회→장바구니 전환율(25%)이 Silver(100%)·Bronze(60%)보다 크게 낮다. Gold 4명 중 3명(C004, C008, C013)이 조회만 하고 장바구니로 넘어가지 않는다. 등급별 합계가 전체 퍼널(15/13/8/6)과 정확히 일치한다.

**인사이트**: 매출/등급이 높은 Gold 고객일수록 오히려 장바구니 전환이 잘 안 된다.

**액션**: Gold 고객에게 조회 후 재방문 시점에 개인화 리마인드(찜한 상품 알림, Gold 전용 쿠폰)를 걸어 장바구니 전환율을 끌어올린다. (Gold 표본이 4명뿐이라 확정적 결론이 아니라 우선순위 신호로 활용)

**셀프체크**
- 왜 이렇게 짰나: `events`와 `customers`를 LEFT JOIN해 등급 미상 고객도 누락 없이 별도 세그먼트로 남김
- 어디서 틀릴 수 있나: grade가 시간에 따라 바뀌는 값이면 조인 시점에 따라 결과가 달라질 수 있음(이 데이터는 고정값이라 문제 없음)
- 왜 믿을 수 있나: 등급별 지표 합계가 전체 퍼널(15/13/8/6)과 정확히 일치하는지 코드로 검산함(아래 셀)

In [3]:
assert df_q1['step1_visit'].sum() == 15
assert df_q1['step2_view'].sum() == 13
assert df_q1['step3_cart'].sum() == 8
assert df_q1['step4_purchase'].sum() == 6
print('등급별 합계가 전체 퍼널(15/13/8/6)과 일치함')


등급별 합계가 전체 퍼널(15/13/8/6)과 일치함


---
## Q2. 첫 구매월 기준 코호트에서 재구매는 보통 몇 개월 뒤 일어나는가? (코호트)

종합 실습 Q2는 **첫 활동월**(events) 기준 코호트 + 1개월 잔존율만 봤다. 여기서는 **첫 구매월**(orders) 기준 코호트로 잡고, 0~4개월 전체 잔존 곡선을 본다.

In [4]:
q2 = """
WITH valid_orders AS (
  SELECT order_id, customer_id, order_date
  FROM orders
  WHERE status NOT IN ('Cancelled', 'Returned')
),
first_purchase AS (
  SELECT customer_id, MIN(date_trunc('month', order_date)) AS cohort_month
  FROM valid_orders
  GROUP BY customer_id
),
cohort_size AS (
  SELECT cohort_month, COUNT(DISTINCT customer_id) AS cohort_customers
  FROM first_purchase
  GROUP BY cohort_month
),
order_activity AS (
  SELECT
    fp.cohort_month,
    date_diff('month', fp.cohort_month, date_trunc('month', vo.order_date)) AS month_index,
    vo.customer_id
  FROM valid_orders vo
  JOIN first_purchase fp ON vo.customer_id = fp.customer_id
)
SELECT
  oa.cohort_month,
  cs.cohort_customers,
  oa.month_index,
  COUNT(DISTINCT oa.customer_id) AS active_customers,
  ROUND(COUNT(DISTINCT oa.customer_id) * 1.0 / NULLIF(cs.cohort_customers, 0) * 100, 1) AS retention_pct
FROM order_activity oa
JOIN cohort_size cs ON oa.cohort_month = cs.cohort_month
GROUP BY oa.cohort_month, cs.cohort_customers, oa.month_index
ORDER BY oa.cohort_month, oa.month_index
"""
df_q2 = con.execute(q2).df()
df_q2


,cohort_month,cohort_customers,month_index,active_customers,retention_pct
0,2023-09-01,4,0,4,100.0
1,2023-09-01,4,1,2,50.0
2,2023-09-01,4,3,2,50.0
3,2023-09-01,4,4,1,25.0
4,2023-10-01,3,0,3,100.0
5,2023-10-01,3,2,1,33.3
6,2023-10-01,3,3,2,66.7
7,2023-11-01,5,0,5,100.0
8,2023-11-01,5,2,1,20.0
9,2023-11-01,5,3,1,20.0


**결과**: 4개 코호트(23-09/10/11/12월 첫구매, 각 4/3/5/2명) 모두 month_index=0은 100%(정의상 당연), **month_index=1(다음 달 재구매)은 4개 코호트 전부 0%**. 재구매가 있더라도 2~4개월 뒤에 발생.

**인사이트**: 첫 구매 직후 1개월 내 재구매가 전혀 발생하지 않는다.

**액션**: 구매 후 30일 시점에 재구매 유도 캠페인(쿠폰, 추천 이메일)을 도입해 재구매 시점을 앞당긴다. (코호트당 2~5명으로 표본이 매우 작아 일반화는 추가 검증 필요)

**셀프체크**
- 왜 이렇게 짰나: 가입일 기준 코호트는 첫 구매까지 4~11개월씩 걸려 매트릭스가 거의 비어 의미가 없었음 → 표준적인 첫 구매월 코호트로 변경
- 어디서 틀릴 수 있나: DuckDB의 `date_diff(part, start, end)`는 BigQuery의 `DATE_DIFF(end, start, part)`와 인자 순서가 반대라 그대로 옮기면 부호가 뒤집힘 — 주의해서 순서를 맞춤

---
## Q3. 실제 매출 기여도(RFM) 상위 고객과 customers.grade가 서로 맞는가? (RFM)

종합 실습 Q3은 고정 임계값(30만원/90일/120일)으로 3등급 **인원수만** 집계했다. 여기서는 NTILE 분위수 기반 RFM 점수를 매기고, 그 결과를 실제 `grade` 컬럼과 **개별 고객 단위로 대조**한다.

In [5]:
q3 = """
WITH valid_orders AS (
  SELECT order_id, customer_id, order_date, amount
  FROM orders
  WHERE status NOT IN ('Cancelled', 'Returned')
),
reference_date AS (
  SELECT MAX(order_date) AS ref_date FROM orders
),
rfm_base AS (
  SELECT
    vo.customer_id,
    date_diff('day', MAX(vo.order_date), (SELECT ref_date FROM reference_date)) AS recency_days,
    COUNT(DISTINCT vo.order_id) AS frequency,
    SUM(vo.amount) AS monetary
  FROM valid_orders vo
  GROUP BY vo.customer_id
),
rfm_scored AS (
  SELECT
    b.customer_id, c.grade,
    b.recency_days, b.frequency, b.monetary,
    NTILE(3) OVER (ORDER BY b.recency_days ASC)  AS r_score,
    NTILE(3) OVER (ORDER BY b.frequency DESC)    AS f_score,
    NTILE(3) OVER (ORDER BY b.monetary DESC)     AS m_score
  FROM rfm_base b
  LEFT JOIN customers c ON b.customer_id = c.customer_id
)
SELECT
  customer_id, grade, recency_days, frequency, monetary, r_score, f_score, m_score,
  CASE
    WHEN r_score = 1 AND f_score = 1 AND m_score = 1 THEN 'VIP'
    WHEN m_score = 1 THEN '고가치 고객'
    WHEN r_score = 3 THEN '휴면 위험군'
    ELSE '일반 고객'
  END AS rfm_segment
FROM rfm_scored
ORDER BY monetary DESC
"""
df_q3 = con.execute(q3).df()
df_q3


,customer_id,grade,recency_days,frequency,monetary,r_score,f_score,m_score,rfm_segment
0,C013,Gold,20,2,725000.0,1,1,1,VIP
1,C007,Silver,39,2,678000.0,1,2,1,고가치 고객
2,C012,Silver,0,2,362000.0,1,1,1,VIP
3,C003,Bronze,27,2,349000.0,1,1,1,VIP
4,C001,Gold,53,3,335000.0,2,1,1,고가치 고객
5,C008,Gold,64,2,314000.0,2,2,2,일반 고객
6,C002,Silver,133,2,247000.0,3,2,2,휴면 위험군
7,C010,Silver,101,1,142000.0,3,3,2,휴면 위험군
8,C005,Silver,70,2,114000.0,2,2,2,일반 고객
9,C015,Bronze,48,1,92000.0,2,2,2,일반 고객


**결과**: 매출(monetary) 상위 3명 C013(725,000/Gold), C007(678,000/Silver), C012(362,000/Silver) 중 VIP로 분류된 건 C013·C012·C003(Bronze)이고, 매출 2위인 C007(Silver)은 f_score가 밀려 'VIP'가 아닌 '고가치 고객'으로 분류됐다. C009는 유효 구매가 없어 이 표에서 아예 빠진다.

**인사이트**: grade(Gold/Silver/Bronze)가 실제 RFM 성과와 잘 맞지 않는다 — 특히 **Bronze 등급인 C003이 VIP로 분류**되고, 매출 2위인 C007은 Silver에 머물러 있다.

**액션**: 등급 산정 로직을 실 구매금액/빈도 기반으로 재검토하고, C003·C007 등 승급 대상을 개별 확인한다. (표본 14명, C009는 미전환 고객으로 별도 관리 필요)

**셀프체크**
- 어디서 틀릴 수 있나: frequency를 order_date로 세면 같은 날 두 번 주문한 고객이 실제보다 줄 수 있어 order_id 기준으로 씀
- 왜 믿을 수 있나: 매출 정렬 순서가 코드 결과와 수기 계산이 정확히 일치하는지 확인함

---
## Q4. 취소·반품율이 특정 국가/등급에 쏠려 있는가? (취소·반품율)

종합 실습 Q4는 **월별** 취소반품율 추이를 봤다. 여기서는 시간축 대신 **국가·등급 세그먼트** 축으로 본다.

In [6]:
q4 = """
WITH order_flags AS (
  SELECT
    o.order_id, o.customer_id, o.status,
    c.country, c.grade,
    CASE WHEN o.status IN ('Cancelled', 'Returned') THEN 1 ELSE 0 END AS is_cancel_or_return
  FROM orders o
  LEFT JOIN customers c ON o.customer_id = c.customer_id
),
by_country AS (
  SELECT COALESCE(country, '미상') AS segment,
         COUNT(*) AS total_orders,
         SUM(is_cancel_or_return) AS cancel_return_orders,
         COUNT(DISTINCT customer_id) AS customers,
         ROUND(SUM(is_cancel_or_return) * 1.0 / NULLIF(COUNT(*), 0) * 100, 1) AS cancel_return_rate_pct
  FROM order_flags GROUP BY country
),
by_grade AS (
  SELECT COALESCE(grade, '미상') AS segment,
         COUNT(*) AS total_orders,
         SUM(is_cancel_or_return) AS cancel_return_orders,
         COUNT(DISTINCT customer_id) AS customers,
         ROUND(SUM(is_cancel_or_return) * 1.0 / NULLIF(COUNT(*), 0) * 100, 1) AS cancel_return_rate_pct
  FROM order_flags GROUP BY grade
)
SELECT 'country' AS dimension, * FROM by_country
UNION ALL
SELECT 'grade' AS dimension, * FROM by_grade
ORDER BY dimension, cancel_return_rate_pct DESC
"""
df_q4 = con.execute(q4).df()
df_q4


,dimension,segment,total_orders,cancel_return_orders,customers,cancel_return_rate_pct
0,country,USA,6,2.0,3,33.3
1,country,Korea,17,4.0,8,23.5
2,country,미상,1,0.0,1,0.0
3,country,Japan,6,0.0,3,0.0
4,grade,Gold,11,3.0,4,27.3
5,grade,Silver,11,2.0,5,18.2
6,grade,Bronze,7,1.0,5,14.3
7,grade,미상,1,0.0,1,0.0


**결과**: 국가별 — USA 33.3%(2/6, 고객 3명) > Korea 23.5%(4/17, 고객 8명) > Japan 0%(0/6, 고객 3명). 등급별 — Gold 27.3%(3/11) > Silver 18.2%(2/11) > Bronze 14.3%(1/7).

**인사이트**: USA·Gold의 취소반품율이 표면적으로 높지만 고객이 3~4명뿐이라 한두 명(특히 C004는 취소 1건+반품 1건)이 비율을 크게 좌우한다.

**액션**: 국가/등급 특성으로 단정하지 말고 C004 등 개별 고객의 취소·반품 사유부터 확인한 뒤, 표본을 늘려 재검증한다.

**셀프체크**
- 어디서 틀릴 수 있나: country가 NULL인 고객(C011)을 빼면 분모가 달라지므로 COALESCE로 '미상' 그룹에 남김

---
## Q5. 재구매까지 평균 며칠 걸리며, 오래 안 산 이탈 위험 고객은 누구인가? (신규 각도 + 이탈위험)

종합 실습 Q5는 이탈위험 **명단**만 뽑았다(경과일>90, 구매≥2회). 여기서는 과제 예시로 제시된 새로운 각도인 **평균 재구매 소요일**을 먼저 구하고, 그 다음 이탈위험 명단을 별도로 뽑는다.

In [7]:
q5a = """
-- (A) 첫 구매 -> 두 번째 구매까지 평균 소요일
WITH valid_orders AS (
  SELECT order_id, customer_id, order_date
  FROM orders
  WHERE status NOT IN ('Cancelled', 'Returned')
),
ranked_orders AS (
  SELECT customer_id, order_date,
         ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date) AS order_seq
  FROM valid_orders
),
first_second AS (
  SELECT f.customer_id,
         date_diff('day', f.order_date, s.order_date) AS days_to_repurchase
  FROM ranked_orders f
  JOIN ranked_orders s ON f.customer_id = s.customer_id AND f.order_seq = 1 AND s.order_seq = 2
)
SELECT
  COUNT(DISTINCT customer_id) AS repeat_customers,
  ROUND(AVG(days_to_repurchase), 1) AS avg_days_to_repurchase,
  MIN(days_to_repurchase) AS min_days,
  MAX(days_to_repurchase) AS max_days
FROM first_second
"""
df_q5a = con.execute(q5a).df()
df_q5a


,repeat_customers,avg_days_to_repurchase,min_days,max_days
0,9,72.9,26,122


In [8]:
q5b = """
-- (B) 마지막 유효 주문 후 60일 넘게 재구매 없는 고객 = 이탈 위험
WITH valid_orders AS (
  SELECT customer_id, order_date
  FROM orders
  WHERE status NOT IN ('Cancelled', 'Returned')
),
reference_date AS (
  SELECT MAX(order_date) AS ref_date FROM orders
),
last_order AS (
  SELECT customer_id, MAX(order_date) AS last_order_date
  FROM valid_orders
  GROUP BY customer_id
)
SELECT
  lo.customer_id, c.name, lo.last_order_date,
  date_diff('day', lo.last_order_date, (SELECT ref_date FROM reference_date)) AS days_since_last_order,
  CASE WHEN date_diff('day', lo.last_order_date, (SELECT ref_date FROM reference_date)) > 60
       THEN '이탈 위험' ELSE '정상' END AS churn_status
FROM last_order lo
JOIN customers c ON lo.customer_id = c.customer_id
ORDER BY days_since_last_order DESC
"""
df_q5b = con.execute(q5b).df()
df_q5b


,customer_id,name,last_order_date,days_since_last_order,churn_status
0,C002,이서연,2023-10-01,133,이탈 위험
1,C010,한예준,2023-11-02,101,이탈 위험
2,C004,최지우,2023-11-08,95,이탈 위험
3,C011,오시우,2023-11-13,90,이탈 위험
4,C005,정하준,2023-12-03,70,이탈 위험
5,C008,윤지호,2023-12-09,64,이탈 위험
6,C014,황지안,2023-12-14,59,정상
7,C001,김민준,2023-12-20,53,정상
8,C015,안수아,2023-12-25,48,정상
9,C007,조은우,2024-01-03,39,정상


**결과**: (A) 재구매 경험 9명, 평균 72.9일(약 2.4개월), 최소 26일~최대 122일. (B) 기준일(2024-02-11, `MAX(order_date)`) 대비 60일 초과 미구매 고객 6명(C002 133일, C010 101일, C004 95일, C011 90일, C005 70일, C008 64일) — 유효 구매 고객 14명 중 43%. 장바구니까지 갔지만 구매를 완료한 적 없는 C009도 별도 위험군.

**인사이트**: 평균 재구매 주기(약 73일)의 절반도 안 되는 시점에 개입해야 이탈 전에 잡을 수 있다.

**액션**: 구매 후 30~40일 시점에 재구매 유도 캠페인을 걸고, 60일 초과 6명에게는 즉시 리마인드/할인 쿠폰을, C009 같은 장바구니 이탈 고객에게는 카트 리마인드 메일을 우선 발송한다.

**셀프체크**
- 왜 이렇게 짰나: 데이터에 '오늘' 개념이 없어 `MAX(order_date)`를 기준일 대리값으로 사용(실서비스라면 오늘 날짜 사용)
- 어디서 틀릴 수 있나: 60일 임계값은 평균 재구매 주기(73일)의 절반 정도로 임의 설정한 값 — 근거를 명시해둠
- 왜 믿을 수 있나: 9명의 첫/두 번째 구매일 차이를 수기로 계산해 합계 656일/9명=72.9일이 코드 결과와 일치함을 확인함

---
## 리포트 한 장으로 엮기

| 질문 | 숫자(진단) | 그래서 무엇을 할 것인가 |
|---|---|---|
| 1. 퍼널(등급별) | Gold 조회→장바구니 25% (Silver 100%, Bronze 60%) | Gold 고객 전용 장바구니 유도 캠페인 |
| 2. 코호트(첫구매월) | 4개 코호트 전부 1개월차 재구매 0% | 구매 후 30일 시점 재구매 유도 캠페인 |
| 3. RFM vs grade | Bronze 고객 C003이 VIP로 분류, 매출 2위 C007은 Silver에 머묾 | 등급 산정 로직을 실 매출 기반으로 재검토 |
| 4. 취소·반품(세그먼트) | USA 33.3%, Gold 27.3% — 단, 표본 3~4명 | 개별 고객(C004 등) 사유 확인 후 재검증 |
| 5. 재구매·이탈위험 | 평균 재구매 72.9일, 60일 초과 미구매 43% | 60일 초과 고객·카트 이탈 고객에 우선 캠페인 |
